# 12 – Bổ sung cho báo cáo: 5 seed, hiệu chuẩn nhiệt độ, thử nghiệm làm nhiễu kênh thu

Ba phần, chạy độc lập (bật/tắt bằng cờ ở ô cấu hình):

1. **Bảng IV với 5 seed** (`RUN_SEEDS5`): lặp lại 4 hàm mục tiêu đã chọn (CE, LS, SORD, Geo-LS) trên XLS-R 13 tầng với seed 0–4, để biết chênh lệch 1–2 điểm có ổn định không. Khoảng 5 phút.
2. **Temperature scaling** (`RUN_TEMP`): mô hình kết hợp đang tự tin quá thấp (ECE ≈ 0,33). Tìm một nhiệt độ T trên tập valid rồi đo lại ECE, Brier, NLL trên test. Accuracy không đổi. Khoảng 3 phút.
3. **Làm nhiễu kênh thu** (`RUN_CHANNEL`): trích lại đặc trưng cho 2.026 câu test sau khi làm nhiễu, rồi chạy các head đã lưu của mô hình kết hợp:
   - `ltas_norm`: cân bằng phổ trung bình dài hạn của mỗi câu về phổ trung bình chung → xóa "màu" kênh thu.
   - `ltas_swap`: áp phổ trung bình của **một tỉnh khác** (chọn ngẫu nhiên) lên câu → nếu mô hình đoán sang tỉnh đó nhiều hơn mức ngẫu nhiên thì nó đang dựa vào màu kênh thu.
   - `ltas_own` (đối chứng): cùng phép xử lý nhưng áp phổ trung bình của **chính tỉnh đúng**. So `ltas_own` với `ltas_swap`/`ltas_norm` tách được tác động của "màu tỉnh" khỏi tác động của bản thân phép cân bằng phổ.

**Chạy lại chỉ phần đối chứng:** đặt `RUN_SEEDS5 = False`, `RUN_TEMP = False`. Các điều kiện đã có cache được nạp lại ngay; báo cáo cũ được giữ và chỉ cập nhật phần mới.
   - `telephone`: dải 300–3400 Hz, 8 kHz, μ-law 8 bit.
   - `noise20`: nhiễu trắng SNR 20 dB.
   Khoảng 30–40 phút trên L4. Đặc trưng được cache trên Drive, nếu bị ngắt chỉ cần chạy lại.

- **Cần GPU** (T4/L4). Cần đã chạy notebook 01, 07 (hoặc 04), 09, 10.
- **Kết quả:** `results/robustness/robustness_report.json`. Gửi file này cho Claude.

In [ ]:
# ==== CẤU HÌNH ====
DRIVE_ROOT = "/content/drive/MyDrive/vimd_dl"
PROVINCE_COL = "province_code"
RUN_SEEDS5, RUN_TEMP, RUN_CHANNEL = True, True, True
SEEDS5 = [0, 1, 2, 3, 4]
N_LAYERS = 13
OBJECTIVES = [("ce", None), ("ls", 0.3), ("sord", 5), ("geols", (0.3, 200.0))]   # tham số đã chọn ở notebook 07
FEAT_B = "full13L_wav2vec2-xls-r-300m_12s"
FEAT_C = "enc_PhoWhisper-large"
XLSR_MODEL, XLSR_KEEP, XLSR_WIN_SEC, MAX_SEC = "facebook/wav2vec2-xls-r-300m", 12, 12.0, 60.0
WHISPER_MODEL, WHISPER_WIN_SEC = "vinai/PhoWhisper-large", 30.0
XLSR_BATCH, WHISPER_BATCH = 16, 8
CHANNEL_CONDS = ["ltas_own", "ltas_norm", "ltas_swap", "telephone", "noise20"]   # điều kiện đã có cache sẽ được nạp lại, không trích lại
PARITY_N = 48            # số câu test dùng để kiểm tra trích đặc trưng lại có khớp với cache không
N_BOOT = 2000

In [ ]:
import os, io, glob, json, time, shutil, warnings
import numpy as np, pandas as pd, pyarrow.parquet as pq, soundfile as sf
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, top_k_accuracy_score
warnings.filterwarnings("ignore")
T0 = time.time()
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
LOCAL = "/content" if os.path.isdir("/content") else os.path.abspath("_work")
LOCAL_DATA, FEAT_DIR, RES_DIR = f"{LOCAL}/vimd16k", f"{DRIVE_ROOT}/features", f"{DRIVE_ROOT}/results"
FIN, OUT = f"{RES_DIR}/final", f"{RES_DIR}/robustness"
os.makedirs(LOCAL_DATA, exist_ok=True); os.makedirs(OUT, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device.type == "cuda" else None
autocast = lambda: torch.autocast(device_type=device.type, dtype=AMP or torch.bfloat16, enabled=AMP is not None)
print("Thiết bị:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

meta = pd.read_csv(f"{DRIVE_ROOT}/metadata.csv", low_memory=False)
meta[PROVINCE_COL] = meta[PROVINCE_COL].astype(str)
prov = pd.read_csv(f"{DRIVE_ROOT}/provinces.csv"); prov[PROVINCE_COL] = prov[PROVINCE_COL].astype(str)
CLASSES = sorted(meta[PROVINCE_COL].unique()); C = len(CLASSES)
y = meta[PROVINCE_COL].map({c: i for i, c in enumerate(CLASSES)}).values
use = ~meta["exclude"].astype(bool).values
tr, va, te = [(meta["split"].values == s) & use for s in ("train", "valid", "test")]
pidx = prov.set_index(PROVINCE_COL)
NAMES = [pidx.loc[c, "province_name"] for c in CLASSES]
region_of = np.array([pidx.loc[c, "region"] for c in CLASSES])
yt, yv = y[te], y[va]; N = len(meta)
GROUPS = {s: (g.index.values, g["row"].values) for s, g in meta.groupby("shard")}

COORD = {
 "CaoBang": (22.666, 106.258), "LangSon": (21.853, 106.761), "QuangNinh": (20.951, 107.080), "HaiPhong": (20.845, 106.688),
 "ThaiBinh": (20.446, 106.336), "NamDinh": (20.420, 106.168), "PhuTho": (21.322, 105.402), "ThaiNguyen": (21.594, 105.848),
 "YenBai": (21.705, 104.875), "TuyenQuang": (21.823, 105.214), "HaGiang": (22.823, 104.984), "LaoCai": (22.486, 103.971),
 "LaiChau": (22.396, 103.458), "SonLa": (21.326, 103.917), "DienBien": (21.386, 103.023), "HoaBinh": (20.817, 105.338),
 "HaNoi": (21.028, 105.854), "HaiDuong": (20.940, 106.333), "NinhBinh": (20.251, 105.975), "VinhPhuc": (21.310, 105.597),
 "HungYen": (20.646, 106.051), "HaNam": (20.541, 105.914), "BacKan": (22.147, 105.835), "BacGiang": (21.273, 106.195),
 "BacNinh": (21.186, 106.076), "ThanhHoa": (19.807, 105.776), "NgheAn": (18.679, 105.682), "HaTinh": (18.343, 105.906),
 "QuangBinh": (17.468, 106.622), "QuangTri": (16.816, 107.100), "ThuaThienHue": (16.464, 107.586), "DaNang": (16.054, 108.202),
 "QuangNam": (15.573, 108.474), "QuangNgai": (15.120, 108.792), "BinhDinh": (13.776, 109.224), "PhuYen": (13.096, 109.309),
 "KhanhHoa": (12.239, 109.197), "NinhThuan": (11.565, 108.988), "BinhThuan": (10.928, 108.102), "KonTum": (14.350, 108.000),
 "GiaLai": (13.983, 108.000), "DakLak": (12.667, 108.038), "DakNong": (12.004, 107.690), "LamDong": (11.940, 108.458),
 "HoChiMinh": (10.776, 106.701), "DongNai": (10.957, 106.843), "BinhDuong": (10.980, 106.651), "BaRiaVungTau": (10.496, 107.169),
 "TayNinh": (11.310, 106.098), "BinhPhuoc": (11.535, 106.883), "LongAn": (10.535, 106.413), "TienGiang": (10.360, 106.360),
 "BenTre": (10.241, 106.376), "VinhLong": (10.254, 105.972), "TraVinh": (9.935, 106.345), "CanTho": (10.045, 105.747),
 "HauGiang": (9.784, 105.470), "SocTrang": (9.603, 105.980), "BacLieu": (9.294, 105.727), "CaMau": (9.176, 105.150),
 "DongThap": (10.460, 105.633), "AnGiang": (10.386, 105.435), "KienGiang": (10.012, 105.081),
}
LAT = np.array([COORD[n][0] for n in NAMES]); LON = np.array([COORD[n][1] for n in NAMES])
p_ = np.pi / 180
DIST = 2 * 6371 * np.arcsin(np.sqrt(np.sin((LAT[None] - LAT[:, None]) * p_ / 2) ** 2 + np.cos(LAT[:, None] * p_)
                                    * np.cos(LAT[None] * p_) * np.sin((LON[None] - LON[:, None]) * p_ / 2) ** 2))

# bootstrap theo người nói trên tập test
spk_te = meta.loc[te, "spk"].astype(str).values
_, spk_id = np.unique(spk_te, return_inverse=True); K = spk_id.max() + 1
B_SPK = np.random.default_rng(0).integers(0, K, (N_BOOT, K)); spk_n = np.bincount(spk_id, minlength=K).astype(float)
def ci_spk(v):
    s = np.bincount(spk_id, weights=v, minlength=K)
    return [round(float(x), 4) for x in np.percentile(s[B_SPK].sum(1) / spk_n[B_SPK].sum(1), [2.5, 97.5])]

def metrics(P, yy):
    pred = P.argmax(1); d = DIST[yy, pred]
    return dict(acc=(pred == yy).mean(), macro_f1=f1_score(yy, pred, average="macro"),
                top5=top_k_accuracy_score(yy, P, k=min(5, C - 1), labels=np.arange(C)),
                region_acc=(region_of[pred] == region_of[yy]).mean(), mean_err_km=d.mean(),
                nll=-np.log(np.clip(P[np.arange(len(yy)), yy], 1e-12, 1)).mean())
REPORT = {}
r4 = lambda x: round(float(x), 4)

## 1. Bảng IV với 5 seed (XLS-R 13 tầng, 12 giây đầu – giống notebook 07)

In [ ]:
if RUN_SEEDS5:
    fdir = sorted(glob.glob(f"{FEAT_DIR}/xlsr_*"))[0]
    first = np.load(f"{fdir}/{next(iter(GROUPS))}.npy")
    XL = np.zeros((N,) + first.shape[1:], np.float16)
    for s, (idx, _) in GROUPS.items():
        XL[idx] = np.load(f"{fdir}/{s}.npy")
    Xg = torch.tensor(XL[:, :N_LAYERS], device=device); del XL
    mtr = torch.tensor(tr, device=device)
    mu, sd = Xg[mtr].float().mean(0), Xg[mtr].float().std(0) + 1e-5
    for i in range(0, len(Xg), 2048):
        Xg[i:i + 2048] = ((Xg[i:i + 2048].float() - mu) / sd).half()
    Xtr_, Xva_, Xte_ = [Xg[torch.tensor(m, device=device)] for m in (tr, va, te)]
    del Xg
    nL, D = Xtr_.shape[1], Xtr_.shape[2]
    ytr_ = torch.tensor(y[tr], device=device)
    DIST_t = torch.tensor(DIST, dtype=torch.float32, device=device); EYE = torch.eye(C, device=device)

    def target_matrix(kind, p):
        if kind == "ce":
            return EYE
        if kind == "ls":
            return (1 - p) * EYE + p / C
        if kind == "sord":
            return torch.softmax(-DIST_t / p, 1)
        eps, tau = p
        nb = torch.exp(-DIST_t / tau) * (1 - EYE)
        return (1 - eps) * EYE + eps * nb / nb.sum(1, keepdim=True)

    class Head7(nn.Module):
        def __init__(s):
            super().__init__(); s.w = nn.Parameter(torch.zeros(nL))
            s.mlp = nn.Sequential(nn.Linear(D, 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, C))
        def forward(s, x):
            return s.mlp((x.float() * s.w.softmax(0)[None, :, None]).sum(1))

    @torch.no_grad()
    def predict7(h, X, bs=2048):
        h.eval()
        return torch.cat([h(X[i:i + bs]).softmax(-1) for i in range(0, len(X), bs)]).cpu().numpy()

    def train7(kind, p, seed, epochs=80, lr=1e-3, wd=1e-3, bs=256, patience=10):
        torch.manual_seed(seed)
        h = Head7().to(device); opt = torch.optim.AdamW(h.parameters(), lr=lr, weight_decay=wd)
        Q = target_matrix(kind, p); best, state, bad = -1, None, 0
        for ep in range(epochs):
            h.train(); perm = torch.randperm(len(Xtr_), device=device)
            for i in range(0, len(perm), bs):
                b = perm[i:i + bs]
                loss = -(Q[ytr_[b]] * F.log_softmax(h(Xtr_[b]), -1)).sum(1).mean()
                opt.zero_grad(); loss.backward(); opt.step()
            acc = (predict7(h, Xva_).argmax(1) == yv).mean()
            if acc > best:
                best, state, bad = acc, {k: v.clone() for k, v in h.state_dict().items()}, 0
            else:
                bad += 1
                if bad >= patience:
                    break
        h.load_state_dict(state)
        return h

    RUNS, P7 = [], {}
    for kind, p in OBJECTIVES:
        for seed in SEEDS5:
            P7[(kind, seed)] = predict7(train7(kind, p, seed), Xte_)
            m = metrics(P7[(kind, seed)], yt)
            RUNS.append(dict(kind=kind, param=str(p), seed=seed, **{k: float(v) for k, v in m.items()}))
        s = pd.DataFrame([r for r in RUNS if r["kind"] == kind])
        print(f"{kind:6s} acc {s.acc.mean():.4f} ± {s.acc.std():.4f} | km {s.mean_err_km.mean():.1f} ± {s.mean_err_km.std():.1f}", flush=True)
    runs5 = pd.DataFrame(RUNS)
    # so với notebook 07 (seed 0–2)
    try:
        old = pd.DataFrame(json.load(open(f"{RES_DIR}/geo_ls_report_13layers.json"))["all_runs"])
        chk = []
        for kind, p in OBJECTIVES:
            o = old[(old.kind == kind) & (old.param == str(p))].set_index("seed")["test_acc"]
            n = runs5[runs5.kind == kind].set_index("seed")["acc"]
            chk += [abs(n[s] - o[s]) for s in o.index if s in n.index]
        print(f"Khớp với notebook 07 (seed 0–2): chênh accuracy lớn nhất {max(chk):.4f}" if chk else "Không tìm thấy run cũ để so.")
    except Exception as e:
        print("Bỏ qua kiểm tra khớp:", e)
    cols = ["acc", "macro_f1", "region_acc", "mean_err_km", "nll"]
    T5 = runs5.groupby("kind")[cols].agg(["mean", "std"]).round(4)
    print(T5.to_string())
    # chênh lệch từng seed (cùng seed) và ensemble 5 seed
    PS = {}
    for a, b in [("geols", "ce"), ("geols", "ls"), ("ls", "ce"), ("sord", "ce")]:
        da = [float((P7[(a, s)].argmax(1) == yt).mean() - (P7[(b, s)].argmax(1) == yt).mean()) for s in SEEDS5]
        dk = [float(DIST[yt, P7[(a, s)].argmax(1)].mean() - DIST[yt, P7[(b, s)].argmax(1)].mean()) for s in SEEDS5]
        ea = np.mean([P7[(a, s)] for s in SEEDS5], 0).argmax(1); eb = np.mean([P7[(b, s)] for s in SEEDS5], 0).argmax(1)
        vacc = (ea == yt).astype(float) - (eb == yt); vkm = DIST[yt, ea] - DIST[yt, eb]
        PS[f"{a} - {b}"] = dict(per_seed_acc=[r4(x) for x in da], per_seed_km=[round(x, 1) for x in dk],
                                seed_mean_acc=r4(np.mean(da)), seed_sd_acc=r4(np.std(da, ddof=1)), n_seeds_positive_acc=int(np.sum(np.array(da) > 0)),
                                seed_mean_km=round(float(np.mean(dk)), 1), seed_sd_km=round(float(np.std(dk, ddof=1)), 1),
                                n_seeds_closer=int(np.sum(np.array(dk) < 0)),
                                ens5_acc_diff=r4(vacc.mean()), ens5_acc_ci_spk=ci_spk(vacc),
                                ens5_km_diff=round(float(vkm.mean()), 1), ens5_km_ci_spk=[round(x * 1, 1) for x in ci_spk(vkm)])
    print(json.dumps(PS, indent=1))
    REPORT["table4_5seeds"] = dict(summary={k: {c: dict(mean=r4(T5.loc[k, (c, "mean")]), sd=r4(T5.loc[k, (c, "std")])) for c in cols} for k in T5.index},
                                   paired=PS, runs=runs5.round(4).to_dict(orient="records"))
    del Xtr_, Xva_, Xte_; torch.cuda.empty_cache()

## 2. Nạp head của mô hình kết hợp D và đặc trưng đã cache (dùng cho mục 3 và 4)

In [ ]:
def load_feat(name):
    first = np.load(f"{FEAT_DIR}/{name}/{next(iter(GROUPS))}.npy")
    X = np.zeros((N,) + first.shape[1:], np.float16)
    for s, (idx, _) in GROUPS.items():
        X[idx] = np.load(f"{FEAT_DIR}/{name}/{s}.npy")
    return X

class HeadD(nn.Module):   # cùng cấu trúc và tên tham số với notebook 10
    def __init__(s, LB, DB, LC, DC):
        super().__init__(); s.keys = ["B", "C"]
        s.register_buffer("mu_B", torch.zeros(LB, DB)); s.register_buffer("sd_B", torch.ones(LB, DB))
        s.register_buffer("mu_C", torch.zeros(LC, DC)); s.register_buffer("sd_C", torch.ones(LC, DC))
        s.w = nn.ParameterDict({"B": nn.Parameter(torch.zeros(LB)), "C": nn.Parameter(torch.zeros(LC))})
        s.mlp = nn.Sequential(nn.Linear(DB + DC, 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, C))
    def forward(s, xb, xc):
        parts = []
        for k, x in (("B", xb), ("C", xc)):
            z = (x.float() - getattr(s, f"mu_{k}")) / getattr(s, f"sd_{k}")
            parts.append((z * s.w[k].softmax(0)[None, :, None]).sum(1))
        return s.mlp(torch.cat(parts, -1))

HEADS = {}
if RUN_TEMP or RUN_CHANNEL:
    XB_all, XC_all = load_feat(FEAT_B), load_feat(FEAT_C)
    LB, DB = XB_all.shape[1:]; LC, DC = XC_all.shape[1:]
    for loss in ("ls", "geols"):
        for s in (0, 1, 2):
            h = HeadD(LB, DB, LC, DC).to(device)
            h.load_state_dict(torch.load(f"{FIN}/head_D_{loss}_seed{s}.pt", map_location=device)); h.eval()
            HEADS[(loss, s)] = h

@torch.no_grad()
def run_heads(xb, xc, loss, bs=512):
    """Trả về xác suất ensemble 3 seed và danh sách xác suất từng seed."""
    outs = []
    for s in (0, 1, 2):
        h = HEADS[(loss, s)]; ps = []
        for i in range(0, len(xb), bs):
            ps.append(h(torch.tensor(xb[i:i + bs], device=device), torch.tensor(xc[i:i + bs], device=device)).softmax(-1).cpu().numpy())
        outs.append(np.concatenate(ps))
    return np.mean(outs, 0), outs

if HEADS:
    CLEAN = {l: run_heads(XB_all[te], XC_all[te], l)[0] for l in ("ls", "geols")}
    ref = {r["model"]: r["test_acc"] for r in json.load(open(f"{FIN}/final_report.json"))["table"]}
    for l in ("ls", "geols"):
        print(f"D {l}: accuracy test từ head đã nạp {(CLEAN[l].argmax(1) == yt).mean():.4f} | notebook 10: {ref[f'D_fusion|{l}']:.4f}")

## 3. Temperature scaling (chỉnh nhiệt độ trên valid)

Dùng log của xác suất ensemble làm logit, tìm T tối thiểu hóa NLL trên valid, áp lên test. Accuracy không đổi vì argmax không đổi.

In [ ]:
def calib(P, yy, bins=15):
    conf = P.max(1); ok = (P.argmax(1) == yy).astype(float); edges = np.linspace(0, 1, bins + 1); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(ok[m].mean() - conf[m].mean())
    Y = np.eye(P.shape[1])[yy]
    return dict(ece=r4(ece), brier=r4(((P - Y) ** 2).sum(1).mean()), nll=r4(-np.log(np.clip(P[np.arange(len(yy)), yy], 1e-12, 1)).mean()),
                mean_conf=r4(conf.mean()), acc=r4(ok.mean()))

def scale(P, T):
    z = np.log(np.clip(P, 1e-12, 1)) / T; z -= z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

if RUN_TEMP:
    TEMP = {}
    for l in ("ls", "geols"):
        Pv, _ = run_heads(XB_all[va], XC_all[va], l)
        grid = np.exp(np.linspace(np.log(0.05), np.log(3.0), 400))
        nlls = [-np.log(np.clip(scale(Pv, T)[np.arange(len(yv)), yv], 1e-12, 1)).mean() for T in grid]
        T = float(grid[int(np.argmin(nlls))])
        before, after = calib(CLEAN[l], yt), calib(scale(CLEAN[l], T), yt)
        TEMP[f"D_fusion|{l}"] = dict(T=round(T, 4), test_before=before, test_after=after, valid_after=calib(scale(Pv, T), yv))
        print(f"D {l}: T = {T:.3f} | test trước {before} | sau {after}")
    REPORT["temperature_scaling"] = TEMP
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(4.2, 3.6)); ax.plot([0, 1], [0, 1], color="#999", lw=1, ls="--")
        for (l, col) in (("geols", "#eb6834"),):
            for P, ls_, lab in ((CLEAN[l], ":", "before"), (scale(CLEAN[l], TEMP[f"D_fusion|{l}"]["T"]), "-", "after")):
                conf = P.max(1); ok = (P.argmax(1) == yt); e = np.linspace(0, 1, 16); xs, ys = [], []
                for lo, hi in zip(e[:-1], e[1:]):
                    m = (conf > lo) & (conf <= hi)
                    if m.sum() >= 10:
                        xs.append(conf[m].mean()); ys.append(ok[m].mean())
                ax.plot(xs, ys, ls_, marker="o", color=col, label=f"Fusion, Geo-LS, {lab} scaling")
        ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy"); ax.legend(frameon=False, fontsize=8)
        fig.tight_layout(); fig.savefig(f"{OUT}/fig_reliability_temp.png", dpi=200); plt.show()
    except Exception as e:
        print("Bỏ qua hình:", e)

## 4. Làm nhiễu kênh thu trên tập test

Bước 4a đọc audio test và tính phổ trung bình dài hạn (LTAS) của từng tỉnh **trên tập valid** (người nói khác tập test).
Bước 4b trích lại đặc trưng XLS-R và PhoWhisper giống hệt notebook 09, có kiểm tra khớp với cache trên vài chục câu chưa làm nhiễu.

In [ ]:
from scipy.signal import stft, istft, butter, sosfiltfilt, resample_poly
NFFT, HOP, SMOOTH = 512, 160, 9

def ensure_local(shards):
    for s in shards:
        src, dst = f"{DRIVE_ROOT}/data16k/{s}", f"{LOCAL_DATA}/{s}"
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
            shutil.copy(src, dst)

def load_waves(mask):
    idx_all = np.where(mask)[0]; pos = {i: k for k, i in enumerate(idx_all)}; out = [None] * len(idx_all)
    sub = meta.loc[mask]
    ensure_local(sub["shard"].unique())
    for s, g in sub.groupby("shard"):
        col = pq.read_table(f"{LOCAL_DATA}/{s}", columns=["audio"]).column(0)
        for i, r in zip(g.index.values, g["row"].values):
            out[pos[i]] = sf.read(io.BytesIO(col[int(r)].as_py()), dtype="float32")[0][:int(MAX_SEC * 16000)]
    return out

def log_ltas(w):
    _, _, Z = stft(w, nperseg=NFFT, noverlap=NFFT - HOP)
    pw = (np.abs(Z) ** 2); e = pw.sum(0); keep = e >= np.percentile(e, 50)      # chỉ lấy nửa số frame có năng lượng cao (tiếng nói)
    l = np.log(pw[:, keep].mean(1) + 1e-10)
    return np.convolve(np.pad(l, SMOOTH // 2, mode="edge"), np.ones(SMOOTH) / SMOOTH, mode="valid")

def apply_gain(w, g_log_power, max_db=15.0):
    g = np.clip(g_log_power / 2, -max_db / 20 * np.log(10), max_db / 20 * np.log(10))   # log power -> log biên độ, giới hạn ±15 dB
    _, _, Z = stft(w, nperseg=NFFT, noverlap=NFFT - HOP)
    _, out = istft(Z * np.exp(g)[:, None], nperseg=NFFT, noverlap=NFFT - HOP)
    out = out[:len(w)].astype(np.float32)
    return out * (np.std(w) / (np.std(out) + 1e-9))

SOS_TEL = butter(4, [300, 3400], btype="bandpass", fs=16000, output="sos")
def telephone(w):
    x = sosfiltfilt(SOS_TEL, w); x = resample_poly(x, 1, 2)
    x = x / (np.abs(x).max() + 1e-9); mu = 255.0
    q = np.round((np.sign(x) * np.log1p(mu * np.abs(x)) / np.log1p(mu) + 1) / 2 * 255) / 255 * 2 - 1
    x = np.sign(q) * (np.expm1(np.abs(q) * np.log1p(mu)) / mu)
    return resample_poly(x, 2, 1)[:len(w)].astype(np.float32)

def add_noise(w, snr_db, rng):
    p = np.mean(w ** 2) + 1e-12
    return (w + rng.standard_normal(len(w)) * np.sqrt(p / 10 ** (snr_db / 10))).astype(np.float32)

if RUN_CHANNEL:
    t = time.time()
    W_te = load_waves(te); W_va = load_waves(va)
    L_va = np.array([log_ltas(w) for w in W_va]); L_te = np.array([log_ltas(w) for w in W_te])
    n_va_prov = np.bincount(yv, minlength=C)
    print("Số câu valid mỗi tỉnh dùng cho LTAS: ít nhất", n_va_prov.min(), "| trung vị", int(np.median(n_va_prov)))
    GLOBAL = L_va.mean(0)
    PROF = np.array([L_va[yv == k].mean(0) if n_va_prov[k] > 0 else GLOBAL for k in range(C)])
    assert np.isfinite(PROF).all() and np.isfinite(L_te).all(), "LTAS có giá trị không hợp lệ"
    rng = np.random.default_rng(0)
    DONOR = np.array([rng.choice([k for k in range(C) if k != c]) for c in yt])
    print(f"Đã đọc {len(W_te)} câu test, {len(W_va)} câu valid, tính LTAS tỉnh | {time.time() - t:.0f}s")

    def perturb(cond, u, w):
        if cond == "clean":
            return w
        if cond == "ltas_norm":
            return apply_gain(w, GLOBAL - L_te[u])
        if cond == "ltas_swap":
            return apply_gain(w, PROF[DONOR[u]] - L_te[u])
        if cond == "ltas_own":      # đối chứng: cùng kiểu xử lý, nhưng áp phổ trung bình của chính tỉnh đúng
            return apply_gain(w, PROF[yt[u]] - L_te[u])
        if cond == "telephone":
            return telephone(w)
        if cond == "noise20":
            return add_noise(w, 20, np.random.default_rng(u))
        raise ValueError(cond)

In [ ]:
def windows(w, win):
    out = [w[i:i + win] for i in range(0, len(w), win)]
    if len(out) > 1 and len(out[-1]) < 2 * 16000:
        out[-2] = np.concatenate([out[-2], out[-1]]); out.pop()
    return out

def extract_list(waves, embed, win_sec, batch):
    items = [(u, wv) for u, w in enumerate(waves) for wv in windows(w, int(win_sec * 16000))]
    items.sort(key=lambda it: -len(it[1])); acc, cnt = None, np.zeros(len(waves))
    for i in range(0, len(items), batch):
        chunk = items[i:i + batch]; pooled, nfr = embed([wv for _, wv in chunk])
        if acc is None:
            acc = np.zeros((len(waves),) + pooled.shape[1:], np.float64)
        for (u, _), pv, n in zip(chunk, pooled, nfr):
            acc[u] += pv * n; cnt[u] += n
    return (acc / cnt[:, None, None]).astype(np.float16)

if RUN_CHANNEL:
    from transformers import AutoModel, AutoFeatureExtractor, WhisperModel
    xenc = AutoModel.from_pretrained(XLSR_MODEL, num_hidden_layers=XLSR_KEEP, layerdrop=0.0)
    if getattr(xenc.config, "do_stable_layer_norm", False):
        xenc.encoder.layer_norm = nn.Identity()
    xenc = xenc.to(device).eval()

    @torch.no_grad()
    def xlsr_embed(ws):
        L = max(len(w) for w in ws)
        x = np.zeros((len(ws), L), np.float32); m = np.zeros((len(ws), L), np.int64)
        for j, w in enumerate(ws):
            w = (w - w.mean()) / (w.std() + 1e-7); x[j, :len(w)] = w; m[j, :len(w)] = 1
        x, m = torch.from_numpy(x).to(device), torch.from_numpy(m).to(device)
        with autocast():
            hs = xenc(x, attention_mask=m, output_hidden_states=True).hidden_states[:XLSR_KEEP + 1]
        fl = xenc._get_feat_extract_output_lengths(m.sum(1)).long()
        fm = (torch.arange(hs[0].shape[1], device=device)[None] < fl[:, None]).float()
        pooled = torch.stack([(h.float() * fm[..., None]).sum(1) / fm.sum(1, keepdim=True) for h in hs], 1)
        return pooled.cpu().numpy(), fl.cpu().numpy()

    wfe = AutoFeatureExtractor.from_pretrained(WHISPER_MODEL)
    wenc = WhisperModel.from_pretrained(WHISPER_MODEL, torch_dtype=torch.float16 if device.type == "cuda" else torch.float32).encoder.to(device).eval()
    CAP = []
    wenc.layers[0].register_forward_pre_hook(lambda mod, args: CAP.append(args[0]))
    for lyr in wenc.layers:
        lyr.register_forward_hook(lambda mod, inp, out: CAP.append(out[0] if isinstance(out, tuple) else out))

    @torch.no_grad()
    def whisper_embed(ws):
        feats = wfe([w for w in ws], sampling_rate=16000, return_tensors="pt").input_features
        CAP.clear(); wenc(feats.to(device, dtype=next(wenc.parameters()).dtype))
        hs = CAP[:len(wenc.layers) + 1]; T = hs[0].shape[1]
        fl = torch.tensor([min(T, int(np.ceil(len(w) / 16000 * 50))) for w in ws], device=device)
        fm = (torch.arange(T, device=device)[None] < fl[:, None]).float()
        pooled = torch.stack([(h.float() * fm[..., None]).sum(1) / fm.sum(1, keepdim=True) for h in hs], 1)
        return pooled.cpu().numpy(), fl.cpu().numpy()

    # 4b. kiểm tra khớp với cache trên PARITY_N câu chưa làm nhiễu
    pi = np.arange(min(PARITY_N, len(W_te)))
    xb_p = extract_list([W_te[u] for u in pi], xlsr_embed, XLSR_WIN_SEC, XLSR_BATCH)
    xc_p = extract_list([W_te[u] for u in pi], whisper_embed, WHISPER_WIN_SEC, WHISPER_BATCH)
    cos = lambda a, b: float(np.mean([np.dot(x.ravel(), z.ravel()) / (np.linalg.norm(x) * np.linalg.norm(z) + 1e-9) for x, z in zip(a.astype(np.float32), b.astype(np.float32))]))
    pr_new = run_heads(xb_p, xc_p, "geols")[0].argmax(1); pr_old = CLEAN["geols"][pi].argmax(1)
    PARITY = dict(n=int(len(pi)), cos_B=round(cos(xb_p, XB_all[te][pi]), 5), cos_C=round(cos(xc_p, XC_all[te][pi]), 5),
                  pred_agreement=r4((pr_new == pr_old).mean()))
    print("Kiểm tra khớp:", PARITY, "(mong đợi cos ≈ 1 và trùng dự đoán ≥ 0.95)")

In [ ]:
if RUN_CHANNEL:
    CH = {}
    base = {l: CLEAN[l] for l in ("ls", "geols")}
    for cond in CHANNEL_CONDS:
        cdir = f"{OUT}/feats_{cond}"; os.makedirs(cdir, exist_ok=True)
        t = time.time()
        if os.path.exists(f"{cdir}/B.npy") and os.path.exists(f"{cdir}/C.npy"):
            xb, xc = np.load(f"{cdir}/B.npy"), np.load(f"{cdir}/C.npy")
        else:
            waves = [perturb(cond, u, w) for u, w in enumerate(W_te)]
            xb = extract_list(waves, xlsr_embed, XLSR_WIN_SEC, XLSR_BATCH); np.save(f"{cdir}/B.npy", xb)
            xc = extract_list(waves, whisper_embed, WHISPER_WIN_SEC, WHISPER_BATCH); np.save(f"{cdir}/C.npy", xc)
            del waves
        res = {}
        for l in ("ls", "geols"):
            P = run_heads(xb, xc, l)[0]; pr, pb = P.argmax(1), base[l].argmax(1)
            dacc = (pr == yt).astype(float) - (pb == yt); dkm = DIST[yt, pr] - DIST[yt, pb]
            r = dict(acc=r4((pr == yt).mean()), acc_clean=r4((pb == yt).mean()), acc_drop=r4(dacc.mean()), acc_drop_ci_spk=ci_spk(dacc),
                     km=round(float(DIST[yt, pr].mean()), 1), km_change=round(float(dkm.mean()), 1),
                     region_acc=r4((region_of[pr] == region_of[yt]).mean()), agreement_with_clean=r4((pr == pb).mean()))
            if cond == "ltas_swap":
                cap = (pr == DONOR).astype(float); cap0 = (pb == DONOR).astype(float)
                r.update(donor_rate=r4(cap.mean()), donor_rate_clean=r4(cap0.mean()), donor_rate_increase_ci_spk=ci_spk(cap - cap0),
                         donor_prob_mean=r4(P[np.arange(len(yt)), DONOR].mean()), donor_prob_mean_clean=r4(base[l][np.arange(len(yt)), DONOR].mean()))
            res[l] = r
        CH[cond] = res
        print(f"[{cond}] {time.time() - t:.0f}s |", json.dumps(res, ensure_ascii=False), flush=True)
    REPORT["channel"] = dict(conditions=CH, parity=PARITY,
                             ltas_profile_spread_db=round(float(10 / np.log(10) * PROF.std(0).mean()), 2),
                             note="ltas gain clipped to ±15 dB; province profiles from validation utterances")

In [ ]:
REPORT["total_minutes"] = round((time.time() - T0) / 60, 1)
path = f"{OUT}/robustness_report.json"
if os.path.exists(path):          # giữ các phần đã chạy trước đó
    old = json.load(open(path)); old.update(REPORT); REPORT = old
json.dump(REPORT, open(path, "w"), indent=1, ensure_ascii=False, default=float)
print(f"Đã lưu {OUT}/robustness_report.json – gửi file này cho Claude. ({REPORT['total_minutes']} phút)")